# Univariate Logistic Regression Feature Screening


This notebook applies **univariate logistic regression** to assess the statistical significance of each predictor against the binary target loan_status.

Univariate logistic regression is a common first step in credit scoring workflows. It evaluates each variable individually to:

- Identify statistically significant features (low p-values)
- Eliminate variables with weak or no association
- Prioritize predictors for further modeling or binning

This approach is especially useful when working with tabular, encoded credit data and aligns with standard model governance practices (e.g., rejecting variables with p > 0.05).

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import datetime

pd.set_option('display.max_columns', None)

## Load Prepared Development Dataset


Loading the dataset generated during the univariate feature preparation phase. This data is clean, encoded, and ready for logistic regression analysis.

In [2]:
data = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_test_0_datasetForUnivariateVSExceptIV.csv', index_col=[0])

In [3]:
data

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,1.002263,63788.0,17,-51.803563,5000.0,33.130018,11.01,0.08,12.0,686,-105.711715,0
11,1.002263,13113.0,0,131.015335,4500.0,-25.457321,8.63,0.34,2.0,651,-105.711715,1
9010,0.369066,59603.0,0,-51.803563,8000.0,11.545480,14.96,0.13,2.0,570,-105.711715,1
7030,-1.125529,54919.0,0,-51.803563,6225.0,55.131699,11.54,0.11,4.0,706,914.813057,0
21143,1.002263,55000.0,7,77.011839,6000.0,11.545480,9.32,0.11,9.0,643,914.813057,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2353,-1.125529,46214.0,3,-51.803563,2500.0,-39.187757,12.53,0.05,2.0,684,-105.711715,0
4958,0.369066,51367.0,4,-51.803563,5000.0,55.131699,10.99,0.10,4.0,569,914.813057,0
38539,-1.125529,88660.0,5,77.011839,12000.0,33.130018,7.46,0.14,4.0,698,-105.711715,0
5937,-0.261705,39754.0,0,-51.803563,5600.0,-39.187757,7.90,0.14,3.0,606,914.813057,0


In [4]:
lstfeature = list(data.columns)

In [5]:
len(lstfeature)

12

In [6]:
lstfeature.remove('loan_status')

In [7]:
len(lstfeature)

11

In [8]:
features = []
pvalues = []
coeff = []

## Univariate Logistic Regression Loop


This loop runs a univariate logistic regression for each feature (one at a time) against the binary target loan_status, capturing their respective p-values.

This method helps:
- Identify statistically significant predictors (low p-values)
- Filter out noisy or irrelevant variables
- Support initial feature ranking before multivariate modeling

In [9]:
for i in lstfeature:
    features.append(i)
    modeldata = data[['loan_status', i]]
    Xtrain = modeldata[[i]]
    Xtrain = sm.add_constant(Xtrain)
    ytrain = modeldata[['loan_status']]

    try:
        log_reg = sm.Logit(ytrain, Xtrain).fit()
        pvalues.append(log_reg.pvalues[i])
        coeff.append(log_reg.params[i])
    except np.linalg.LinAlgError as err:
        if 'Singular matrix' in str(err):
            pvalues.append(1.0)
            coeff.append(0.0)
            continue
        else:
            raise

Optimization terminated successfully.
         Current function value: 0.529700
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.502176
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.529479
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.494564
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.523187
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.519482
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.469969
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.460272
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.529589
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.529672
  

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [10]:
dict = {'features': features, 'pvalues': pvalues}
df = pd.DataFrame(dict)

In [11]:
df

,features,pvalues
0,person_education,5.322431e-01
1,person_income,4.827454e-285
2,person_emp_exp,1.538676e-04
3,person_home_ownership,0.000000e+00
4,loan_amnt,3.439754e-96
5,loan_intent,2.368039e-140
6,loan_int_rate,0.000000e+00
7,loan_percent_income,0.000000e+00
8,cb_person_cred_hist_length,6.212274e-03
9,credit_score,1.368452e-01


## Rank Features by Statistical Significance

Sorting variables by their p-value gives a sense of which predictors are statistically strongest. This helps shortlist variables for further modeling or transformation.


In [12]:
df = df.sort_values(by = ['pvalues'], ascending = True).reset_index().drop(columns = ['index'])

## Interpretation of Results

The table ranks each feature based on its univariate logistic regression p-value against the target loan_status.

- Features like person_income, loan_int_rate, and loan_percent_income show **strong statistical significance** (p < 0.001), suggesting a meaningful relationship with default likelihood.
- Variables such as credit_score, person_education, and especially previous_loan_defaults_on_file exhibit **weak or no statistical significance**, with p-values above common rejection thresholds (e.g., 0.05).
- This step helps **filter out irrelevant predictors** and prioritizes those for further smoothing, binning, or multivariate analysis.

> Note: While statistical significance is important, it does not guarantee predictive power, further validation through model performance and multicollinearity checks is recommended.


In [13]:
df

,features,pvalues
0,person_home_ownership,0.000000e+00
1,loan_percent_income,0.000000e+00
2,loan_int_rate,0.000000e+00
3,person_income,4.827454e-285
4,loan_intent,2.368039e-140
5,loan_amnt,3.439754e-96
6,person_emp_exp,1.538676e-04
7,cb_person_cred_hist_length,6.212274e-03
8,credit_score,1.368452e-01
9,person_education,5.322431e-01


In [14]:
df.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_LR.csv')